In [134]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt


In [135]:
def get_sp500_tickers():
  table = pd.read_html("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies")
  return table[0]['Symbol'].str.replace('.', '-', regex=False).tolist()

In [136]:
def download_sp500_data():
  tickers = get_sp500_tickers()

  data = yf.download(
    tickers=tickers,
    period="730d",
    interval="4h",
    group_by="ticker",
    auto_adjust=True,
    threads=True,
    progress=True
  )

  return data

In [137]:
def add_moving_averages(df: pd.DataFrame, spans = [5, 10, 20]):
  for span in spans:
    df[f"MA{span}"] = df["Close"].rolling(window=span*2).mean()
  
  return df

In [138]:
def add_rsi(df: pd.DataFrame):
  delta = df["Close"].diff()

  up = delta.clip(lower=0)
  down = -1 * delta.clip(upper=0)

  avg_gain = up.rolling(window=14*2).mean()
  avg_loss = down.rolling(window=14*2).mean()

  rs = avg_gain / avg_loss

  df["RSI14"] = 100 - (100 / (rs + 1))

  return df

In [139]:
def add_standard_deviations(df: pd.DataFrame, spans=[5, 10, 20]):
  for span in spans:
    df[f"SD{span}"] = df["Close"].rolling(window=span*2).std()

  return df

In [140]:
def add_rate_of_change(df: pd.DataFrame, spans=[5, 10, 20]):
  for span in spans:
    df[f"ROC{span}"] = df["Close"].pct_change(periods=span*2)

  return df

In [141]:
def add_avg_true_range(df: pd.DataFrame):
  df["HighLow"] = df["High"] - df["Low"]
  df["HighClose"] = (df["High"] - df["Close"].shift()).abs()
  df["LowClose"] = (df["Low"] - df["Close"].shift()).abs()

  df["TR"] = df[[
    "HighLow",
    "HighClose",
    "LowClose"
  ]].max(axis=1)

  df["ATR14"] = df["TR"].rolling(window=14*2).mean()

  df.drop(
    columns=[
      "HighLow",
      "HighClose",
      "LowClose",
      "TR"
    ],
    inplace=True
  )

  return df

In [142]:
def add_bollinger_bands(df: pd.DataFrame, window=20, no_std=2):
  rolling_mean = df["Close"].rolling(window=window*2).mean()
  rolling_std = df["Close"].rolling(window=window*2).std()

  df["BollMid"] = rolling_mean
  df["BollUpper"] = rolling_mean + (no_std * rolling_std)
  df["BollLower"] = rolling_mean - (no_std * rolling_std)

  df["BollBandwidth"] = df["BollUpper"] - df["BollLower"]

  return df

In [143]:
def add_ema(df: pd.DataFrame, spans=[5, 12, 26, 50]):
  for span in spans:
    df[f"EMA{span}"] = df["Close"].ewm(span=span*2).mean()

  return df

In [144]:
def add_macd(df: pd.DataFrame, short_window=12, long_window=26, signal_window=9):
  short_ema = df["Close"].ewm(span=short_window*2, adjust=False).mean()
  long_ema = df["Close"].ewm(span=long_window*2, adjust=False).mean()

  df["MACD"] = short_ema - long_ema
  df["MACD_Signal"] = df["MACD"].ewm(span=signal_window*2, adjust=False).mean()
  df["MACD_Histo"] = df["MACD"] - df["MACD_Signal"]

  return df

In [145]:
def add_donchian_channels(df: pd.DataFrame, window=20):
    df["DC_Lower"] = df["Low"].rolling(window=window*2).min()
    df["DC_Upper"] = df["High"].rolling(window=window*2).max()
    df["DC_Mid"] = (df["High"] + df["Low"]) / 2
    df["DC_Width"] = df["DC_Upper"] - df["DC_Lower"]
    
    df["DC_Breakout_Strength"] = (df["Close"] - df["DC_Mid"]) / df["DC_Width"]
    
    return df

In [146]:
def add_volume_features(df: pd.DataFrame):
  df["Vol_MA10"] = df["Volume"].rolling(window=10*2).mean()
  df["Vol_MA20"] = df["Volume"].rolling(window=20*2).mean()
  
  df["Vol_MA10_Ratio"] = df["Volume"] / df["Vol_MA10"]
  df["Vol_MA20_Ratio"] = df["Volume"] / df["Vol_MA20"]
  
  df["Vol_Spike"] = (df["Vol_MA10_Ratio"] > 1.5).astype(int)
  
  vol_mean = df["Volume"].rolling(window=20*2).mean()
  vol_std = df["Volume"].rolling(window=20*2).std()
  
  df["Vol_Z"] = (df["Volume"] - vol_mean) / vol_std
  
  return df

In [147]:
def add_mfi(df: pd.DataFrame, period=14):
  typical_price = (df["High"] + df["Low"] + df["Close"]) / 3

  raw_money_flow = typical_price * df["Volume"]

  delta_tp = typical_price.diff()

  pos_flow = raw_money_flow.where(delta_tp > 0, 0.0)
  neg_flow = raw_money_flow.where(delta_tp < 0, 0.0)

  pos_sum = pos_flow.rolling(window=period*2).sum()
  neg_sum = neg_flow.rolling(window=period*2).sum()

  money_flow_ratio = pos_sum / (neg_sum + 1e-9) # avoid div by zero

  money_flow_index = 100 - (100 / (1 + money_flow_ratio))

  df["MFI"] = money_flow_index

  return df

In [148]:
def add_z_close(df: pd.DataFrame):
  z_mean = df["Close"].rolling(window=20*2).mean()
  z_std = df["Close"].rolling(window=20*2).std()

  df["ZClose20"] = (df["Close"] - z_mean) / z_std

  return df

In [149]:
def add_stochastic_oscillator(df: pd.DataFrame, k_period=14, d_period=3):
  high_max = df["High"].rolling(window=k_period*2).max()
  low_min = df["Low"].rolling(window=k_period*2).min()

  df["Stoch_K%"] = 100 * (
    (df["Close"] - low_min)
    /
    (high_max - low_min + 1e-9)
  ) # avoid div by zero
  
  df["Stoch_D%"] = df["Stoch_K%"].rolling(window=d_period*2).mean()

  return df

In [150]:
FEATURES = [
  add_moving_averages,
  add_rsi,
  add_standard_deviations,
  add_rate_of_change,
  add_avg_true_range,
  add_bollinger_bands,
  add_macd,
  add_ema,
  add_donchian_channels,
  add_volume_features,
  add_mfi,
  add_z_close,
  add_stochastic_oscillator
]

In [151]:
def derive_features(df: pd.DataFrame) -> pd.DataFrame:
  for f in FEATURES:
    df = f(df)

  return df

In [ ]:
def process_ticker(ticker, interval="4h", period="730d"):
  try:
    df = yf.download(
      ticker,
      interval=interval,
      period=period,
      auto_adjust=True,
      progress=True
    )

    if df.empty or len(df) < 50:
      return None
    
    if isinstance(df.columns, pd.MultiIndex):
      df.columns = df.columns.get_level_values(0)

    df = derive_features(df=df)

    df.dropna(inplace=True)

    df.columns = pd.MultiIndex.from_product([df.columns, [ticker]])

    return df

  except Exception as e:
    print(f"Error for ticker ${ticker}: {e}")
    return None

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume,MA5,MA10,MA20,RSI14,SD5,...,Vol_MA10,Vol_MA20,Vol_MA10_Ratio,Vol_MA20_Ratio,Vol_Spike,Vol_Z,MFI,ZClose20,Stoch_K%,Stoch_D%
,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,...,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL
Datetime,,,,,,,,,,,,,,,,,,,,,
2022-06-30 13:30:00+00:00,137.419998,138.369995,133.773697,137.250000,51637683,139.611041,136.832975,139.790092,41.117296,1.760896,...,35868485.10,3.552330e+07,1.439639,1.453629,0,1.190936,51.184737,-0.390839,49.265139,50.658697
2022-06-30 17:30:00+00:00,136.759995,138.149994,135.699997,137.429993,27737291,139.462041,136.898475,139.491592,43.613028,1.942530,...,35143485.35,3.564977e+07,0.789258,0.778050,0,-0.589121,53.765386,-0.462504,53.678203,49.467291
2022-07-01 13:30:00+00:00,137.050003,138.320007,135.660004,137.000000,39301850,139.166042,137.188475,139.163365,48.746940,2.071026,...,34302588.25,3.550436e+07,1.145740,1.106958,0,0.284304,58.254460,-0.373618,55.694672,50.776024
2022-07-01 17:30:00+00:00,138.940002,139.039993,137.020004,137.059998,22782029,138.882042,137.630475,138.856115,52.145317,1.856343,...,33566618.30,3.557576e+07,0.678711,0.640381,0,-0.963447,60.889073,0.015806,68.836091,55.527530
2022-07-05 13:30:00+00:00,139.839996,140.029999,136.929993,137.770004,40102209,138.650041,138.000020,138.718365,58.264116,1.514687,...,32216314.05,3.520010e+07,1.244780,1.139264,0,0.379435,66.529569,0.215496,75.093869,59.426766
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-04-25 17:30:00+00:00,209.229996,209.429993,207.369995,209.389999,9828701,202.491917,200.779414,201.285294,59.652630,6.728218,...,23808784.35,3.259682e+07,0.412818,0.301523,0,-0.964328,56.139776,0.596722,91.516088,83.895488
2025-04-28 13:30:00+00:00,208.139999,211.500000,207.460007,210.059998,18061129,204.244518,201.344878,201.026544,63.007136,5.452013,...,22277445.55,3.255200e+07,0.810736,0.554839,0,-0.613063,63.399809,0.544288,89.023523,88.019324
